# Day 9 Solution: Resume → Structured JSON Parser

Demonstrates all five Day 9 concepts:
1. Schema-guided extraction (Lesson 1)
2. Optional fields, nested models, `Field(description=...)` (Lesson 2)
3. Few-shot grounding — two example pairs (Lesson 3)
4. Extract-validate-retry loop (Lesson 4)
5. Section-aware extraction — split, extract per section, assemble (Lesson 5)

**Deliverable:** Run all cells. The final cell pretty-prints a fully validated
`ResumeProfile` as indented JSON — produced entirely by a local Ollama model.

## Cell 1 — Imports and configuration

In [ ]:
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

# Day 5 logging pattern: debug for calls, info for success, warning for retries
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)
logger = logging.getLogger(__name__)

MODEL = "llama3.2"

## Cell 2 — Pydantic schemas (Lessons 1 & 2)

Four models compose the full resume structure. Every field carries
`Field(description=...)` — that description travels into the JSON Schema
embedded in the system prompt and guides the model's extraction choices.

In [ ]:
class PersonalInfo(BaseModel):
    """Contact details extracted from the resume preamble."""
    name: str = Field(
        description="Full name of the candidate exactly as written"
    )
    email: str = Field(
        description="Primary email address in lowercase"
    )
    phone: str | None = Field(
        default=None,
        description="Phone number in any format, or null if not present"
    )
    location: str | None = Field(
        default=None,
        description="City and region or country (e.g. 'Chicago, IL'), or null if not stated"
    )


class WorkExperience(BaseModel):
    """One entry in the work history section."""
    company: str = Field(
        description="Name of the employer or organisation"
    )
    title: str = Field(
        description="Job title held at this company"
    )
    start_year: int = Field(
        description="Four-digit year the role started (e.g. 2019)"
    )
    end_year: int | None = Field(
        default=None,
        description="Four-digit year the role ended, or null if it is the current role"
    )
    responsibilities: list[str] = Field(
        default_factory=list,
        description="Key responsibilities or achievements, each as a short phrase or sentence"
    )


class Education(BaseModel):
    """One education or certification entry."""
    institution: str = Field(
        description="Name of the university, college, or certifying body"
    )
    degree: str = Field(
        description="Degree, diploma, or certificate title"
    )
    graduation_year: int | None = Field(
        default=None,
        description="Four-digit graduation or completion year, or null if not stated"
    )


class ResumeProfile(BaseModel):
    """Complete structured representation of a plain-text resume."""
    personal: PersonalInfo = Field(
        description="Contact and personal details of the candidate"
    )
    work: list[WorkExperience] = Field(
        default_factory=list,
        description="Work history entries in reverse chronological order"
    )
    education: list[Education] = Field(
        default_factory=list,
        description="Education and certification entries"
    )
    skills: list[str] = Field(
        default_factory=list,
        description="Technical and professional skills, each as a short phrase"
    )


logger.debug("Schemas defined: PersonalInfo, WorkExperience, Education, ResumeProfile")

## Cell 3 — Section splitter (Lesson 5)

A pure-Python line scanner — no model calls. Splits the document into labelled
chunks so each section can be extracted with a focused schema.

In [ ]:
def split_sections(
    document: str,
    section_names: list[str],
) -> dict[str, str]:
    """Split a document into labelled sections by detecting header lines.

    A header is a line whose stripped, upper-cased content exactly matches
    one of the names in section_names (case-insensitive). Text before the
    first recognised header is stored under the key 'PREAMBLE'.

    Args:
        document:      Full document text.
        section_names: Section headers to detect, e.g. ['SKILLS', 'EXPERIENCE'].

    Returns:
        dict mapping upper-cased section label to stripped section text.
        Empty sections are omitted.
    """
    # Normalise once; comparison is then O(1) per line
    name_set = {name.strip().upper() for name in section_names}

    sections: dict[str, list[str]] = {"PREAMBLE": []}
    current = "PREAMBLE"

    for line in document.splitlines():
        stripped_upper = line.strip().upper()
        if stripped_upper in name_set:
            # This line is a section header — switch context
            current = stripped_upper
            sections.setdefault(current, [])
        else:
            sections.setdefault(current, []).append(line)

    # Join lines; drop sections that are entirely blank
    return {
        label: "\n".join(lines).strip()
        for label, lines in sections.items()
        if "\n".join(lines).strip()
    }


# Quick smoke test on a tiny document
_test_doc = "Alice\nalice@x.com\nSKILLS\nPython\nEXPERIENCE\nEngineer at Acme, 2020-2023."
_sections = split_sections(_test_doc, ["SKILLS", "EXPERIENCE"])
assert "PREAMBLE" in _sections
assert "SKILLS" in _sections
assert "EXPERIENCE" in _sections
logger.debug("split_sections smoke test passed: %s", list(_sections.keys()))

## Cell 4 — Extract-validate-retry loop (Lesson 4)

Generic function: works with any Pydantic schema. On `ValidationError` it
appends the failed output and the error details as a repair turn, then retries.

In [ ]:
def _build_system_prompt(schema: dict) -> str:
    """System prompt that embeds the JSON Schema as the extraction contract."""
    return (
        "You are a precise data extraction assistant.\n"
        "Extract information from the provided text and return a JSON object "
        "that matches this schema exactly:\n\n"
        f"{json.dumps(schema, indent=2)}\n\n"
        "Rules:\n"
        "- Return ONLY the JSON object — no prose, no markdown, no explanation.\n"
        "- Set optional fields to null when the information is absent from the text.\n"
        "- Do not invent values that are not in the text."
    )


def extract_with_retry(
    text: str,
    model_class: type[BaseModel],
    messages_override: list[dict] | None = None,
    max_retries: int = 3,
) -> BaseModel:
    """Extract structured data from text, retrying on ValidationError.

    Args:
        text:             Source text to extract from.
        model_class:      Pydantic BaseModel subclass defining the target schema.
        messages_override: If provided, use a copy of this messages list (supports
                          few-shot injection). Otherwise builds a default 2-message list.
        max_retries:      Maximum correction attempts (default 3).

    Returns:
        A validated instance of model_class.

    Raises:
        ValidationError: If all attempts are exhausted without valid output.
    """
    schema = model_class.model_json_schema()

    if messages_override is not None:
        # Few-shot path — copy to avoid mutating the caller's list across retries
        messages = list(messages_override)
    else:
        # Zero-shot path — build a plain 2-message list
        messages = [
            {"role": "system", "content": _build_system_prompt(schema)},
            {"role": "user",   "content": text},
        ]

    last_error: ValidationError | None = None

    for attempt in range(1, max_retries + 1):
        logger.debug(
            "extract_with_retry | %s | attempt %d/%d",
            model_class.__name__, attempt, max_retries,
        )
        response = ollama.chat(model=MODEL, messages=messages, format="json")
        raw = response["message"]["content"]
        logger.debug("Raw response (%d chars): %s", len(raw), raw[:120])

        try:
            result = model_class.model_validate_json(raw)
            logger.info(
                "extract_with_retry | %s | success on attempt %d",
                model_class.__name__, attempt,
            )
            return result

        except ValidationError as exc:
            last_error = exc
            logger.warning(
                "extract_with_retry | %s | attempt %d validation failed: %s",
                model_class.__name__, attempt, exc.errors(),
            )
            if attempt < max_retries:
                # Feed the failed output and the error back as a repair turn
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": (
                        f"Your JSON failed validation with these errors:\n\n{exc}\n\n"
                        "Return the corrected JSON only. "
                        "Use the original text above to find the correct values. "
                        "Do not add fields not in the schema."
                    ),
                })

    raise last_error  # type: ignore[misc]

## Cell 5 — Few-shot message builder (Lesson 3)

Wraps the system prompt + example pairs + real input into a messages list.
The examples show the model both the fully-populated and the sparse case so it
knows when to fill a field and when to return null.

In [ ]:
def build_fewshot_messages(
    schema: dict,
    examples: list[tuple[str, dict]],
    real_input: str,
) -> list[dict]:
    """Build the full messages list: system + example pairs + real input.

    Args:
        schema:      JSON Schema dict from model_json_schema().
        examples:    List of (input_prose, output_dict) demonstration pairs.
        real_input:  The actual text to extract from.

    Returns:
        A messages list ready to pass to ollama.chat().
    """
    messages: list[dict] = [
        {"role": "system", "content": _build_system_prompt(schema)}
    ]

    # Inject each example as a user/assistant demonstration pair
    for input_prose, output_dict in examples:
        messages.append({"role": "user",      "content": input_prose})
        messages.append({"role": "assistant", "content": json.dumps(output_dict)})

    # The real extraction request comes last
    messages.append({"role": "user", "content": real_input})
    return messages

## Cell 6 — Per-section helper schemas and few-shot examples

Each section of the resume needs a small wrapper schema so the list fields are
at the top level of the JSON the model produces. The few-shot examples are
fabricated — realistic but not real personal data.

In [ ]:
# --- Wrapper schemas used for per-section extraction ---
# Each wraps the list so the model returns a clean top-level JSON object.

class _SkillsSection(BaseModel):
    skills: list[str] = Field(
        default_factory=list,
        description="Each technical or professional skill as a short phrase"
    )


class _WorkSection(BaseModel):
    jobs: list[WorkExperience] = Field(
        default_factory=list,
        description="Work history entries in the order they appear in the text"
    )


class _EducationSection(BaseModel):
    entries: list[Education] = Field(
        default_factory=list,
        description="Education and certification entries in the order they appear"
    )


# --- Few-shot examples for PersonalInfo (Lesson 3) ---
# Two examples: one fully populated, one sparse (only required fields)
PERSONAL_EXAMPLES: list[tuple[str, dict]] = [
    (
        # Example 1 — all fields present
        "Maria Santos\nmaria.santos@brighttech.io\n+44-20-7946-0821\nLondon, UK",
        {
            "name": "Maria Santos",
            "email": "maria.santos@brighttech.io",
            "phone": "+44-20-7946-0821",
            "location": "London, UK",
        },
    ),
    (
        # Example 2 — only name and email present
        "David Okafor\nd.okafor@example.com",
        {
            "name": "David Okafor",
            "email": "d.okafor@example.com",
            "phone": None,
            "location": None,
        },
    ),
]


# --- Few-shot examples for _SkillsSection ---
SKILLS_EXAMPLES: list[tuple[str, dict]] = [
    (
        "Python, Rust, PostgreSQL, Kubernetes, CI/CD pipelines",
        {"skills": ["Python", "Rust", "PostgreSQL", "Kubernetes", "CI/CD pipelines"]},
    ),
    (
        "Excel, data analysis",
        {"skills": ["Excel", "data analysis"]},
    ),
]


# --- Few-shot examples for _WorkSection ---
WORK_EXAMPLES: list[tuple[str, dict]] = [
    (
        # One current role, one past role
        "Lead Engineer, Nexus Systems, 2022-present.\n"
        "Architected microservices platform. Reduced deployment time by 60 percent.\n"
        "Software Engineer, BlueSky Ltd, 2019-2022.\n"
        "Developed REST APIs in Python. Maintained CI/CD pipelines.",
        {
            "jobs": [
                {
                    "company": "Nexus Systems",
                    "title": "Lead Engineer",
                    "start_year": 2022,
                    "end_year": None,
                    "responsibilities": [
                        "Architected microservices platform",
                        "Reduced deployment time by 60 percent",
                    ],
                },
                {
                    "company": "BlueSky Ltd",
                    "title": "Software Engineer",
                    "start_year": 2019,
                    "end_year": 2022,
                    "responsibilities": [
                        "Developed REST APIs in Python",
                        "Maintained CI/CD pipelines",
                    ],
                },
            ]
        },
    ),
    (
        # One entry with minimal information
        "Intern, Small Co, 2017-2018.",
        {
            "jobs": [
                {
                    "company": "Small Co",
                    "title": "Intern",
                    "start_year": 2017,
                    "end_year": 2018,
                    "responsibilities": [],
                }
            ]
        },
    ),
]


# --- Few-shot examples for _EducationSection ---
EDUCATION_EXAMPLES: list[tuple[str, dict]] = [
    (
        "MSc Artificial Intelligence, University of Edinburgh, 2020.\n"
        "BSc Mathematics, University of Cape Town, 2018.",
        {
            "entries": [
                {"institution": "University of Edinburgh",
                 "degree": "MSc Artificial Intelligence",
                 "graduation_year": 2020},
                {"institution": "University of Cape Town",
                 "degree": "BSc Mathematics",
                 "graduation_year": 2018},
            ]
        },
    ),
    (
        "Certificate in Cloud Architecture, AWS, 2021.",
        {
            "entries": [
                {"institution": "AWS",
                 "degree": "Certificate in Cloud Architecture",
                 "graduation_year": 2021},
            ]
        },
    ),
]

logger.debug("Section schemas and few-shot examples ready")

## Cell 7 — The plain-text resume (source document)

In [ ]:
RESUME_TEXT = """
Jordan Rivera
jordan.rivera@example.com
+1-312-555-0174
Chicago, IL

SKILLS
Python, SQL, Apache Spark, dbt, Airflow, data modelling, REST APIs, Docker

EXPERIENCE
Senior Data Engineer, Luminary Analytics, 2021-present.
Designed and maintained ETL pipelines processing 50M events/day using Python
and Apache Spark. Led migration from on-prem Hadoop to AWS EMR.

Data Engineer, GreenPath Technologies, 2018-2021.
Built dbt models for financial reporting. Wrote Airflow DAGs for nightly batch
jobs. Reduced pipeline failure rate by 40 percent through improved error handling.

Junior Analyst, DataFirst Consulting, 2016-2018.
Delivered SQL-based reports for retail clients. Automated weekly Excel
summaries with Python scripts.

EDUCATION
BSc Computer Science, University of Illinois at Chicago, 2016.
Certificate in Data Engineering, Coursera / Google, 2019.
"""

print("Resume loaded:", len(RESUME_TEXT), "characters")

## Cell 8 — Section-aware extraction pipeline (Lesson 5)

`parse_resume` orchestrates all five concepts:
1. Split the document (Lesson 5)
2. Extract each section with its own focused schema (Lessons 1 & 2)
3. Use few-shot examples for grounding (Lesson 3)
4. Retry on ValidationError (Lesson 4)
5. Assemble one `ResumeProfile` from all section results

In [ ]:
def parse_resume(document: str) -> ResumeProfile:
    """Parse a plain-text resume into a fully validated ResumeProfile.

    Pipeline:
      1. split_sections — divide the document into labelled text chunks
      2. Extract PREAMBLE → PersonalInfo  (few-shot + retry)
      3. Extract SKILLS   → _SkillsSection (few-shot + retry)
      4. Extract EXPERIENCE → _WorkSection (few-shot + retry)
      5. Extract EDUCATION  → _EducationSection (few-shot + retry)
      6. Assemble and return ResumeProfile

    Args:
        document: Full plain-text resume string.

    Returns:
        A validated ResumeProfile instance.
    """
    logger.info("parse_resume | starting section-aware extraction")

    # --- Stage 1: Split ---
    sections = split_sections(document, ["SKILLS", "EXPERIENCE", "EDUCATION"])
    logger.debug("parse_resume | detected sections: %s", list(sections.keys()))

    # --- Stage 2: Extract PREAMBLE → PersonalInfo ---
    preamble_text = sections.get("PREAMBLE", "")
    personal_messages = build_fewshot_messages(
        schema=PersonalInfo.model_json_schema(),
        examples=PERSONAL_EXAMPLES,
        real_input=preamble_text,
    )
    personal_info: PersonalInfo = extract_with_retry(
        text=preamble_text,
        model_class=PersonalInfo,
        messages_override=personal_messages,
        max_retries=3,
    )
    logger.info("parse_resume | PersonalInfo extracted: %s", personal_info.name)

    # --- Stage 3: Extract SKILLS ---
    skills_text = sections.get("SKILLS", "")
    skills_messages = build_fewshot_messages(
        schema=_SkillsSection.model_json_schema(),
        examples=SKILLS_EXAMPLES,
        real_input=skills_text,
    )
    skills_section: _SkillsSection = extract_with_retry(
        text=skills_text,
        model_class=_SkillsSection,
        messages_override=skills_messages,
        max_retries=3,
    )
    logger.info("parse_resume | Skills extracted: %d items", len(skills_section.skills))

    # --- Stage 4: Extract EXPERIENCE ---
    experience_text = sections.get("EXPERIENCE", "")
    work_messages = build_fewshot_messages(
        schema=_WorkSection.model_json_schema(),
        examples=WORK_EXAMPLES,
        real_input=experience_text,
    )
    work_section: _WorkSection = extract_with_retry(
        text=experience_text,
        model_class=_WorkSection,
        messages_override=work_messages,
        max_retries=3,
    )
    logger.info(
        "parse_resume | Work experience extracted: %d entries",
        len(work_section.jobs),
    )

    # --- Stage 5: Extract EDUCATION ---
    education_text = sections.get("EDUCATION", "")
    education_messages = build_fewshot_messages(
        schema=_EducationSection.model_json_schema(),
        examples=EDUCATION_EXAMPLES,
        real_input=education_text,
    )
    education_section: _EducationSection = extract_with_retry(
        text=education_text,
        model_class=_EducationSection,
        messages_override=education_messages,
        max_retries=3,
    )
    logger.info(
        "parse_resume | Education extracted: %d entries",
        len(education_section.entries),
    )

    # --- Stage 6: Assemble ---
    # Validate the assembled ResumeProfile to confirm all nested objects are sound
    profile = ResumeProfile(
        personal=personal_info,
        work=work_section.jobs,
        education=education_section.entries,
        skills=skills_section.skills,
    )
    logger.info("parse_resume | ResumeProfile assembled successfully")
    return profile

## Cell 9 — Run the parser and print the deliverable

In [ ]:
print("Running section-aware resume parser...")
print("-" * 60)

# Run the full pipeline
profile = parse_resume(RESUME_TEXT)

# Pretty-print as indented JSON — this is the deliverable
output_json = json.dumps(profile.model_dump(), indent=2)
print(output_json)

print("-" * 60)
print(f"Candidate : {profile.personal.name}")
print(f"Email     : {profile.personal.email}")
print(f"Skills    : {len(profile.skills)} extracted")
print(f"Work      : {len(profile.work)} role(s)")
print(f"Education : {len(profile.education)} entry/entries")

## Cell 10 — Gate check

In [ ]:
import requests

# Confirm Ollama is reachable
try:
    requests.get("http://localhost:11434/api/tags", timeout=3)
except Exception as e:
    raise AssertionError(f"Ollama server is not running: {e}") from e

# Confirm the deliverable is a fully validated ResumeProfile
assert isinstance(profile, ResumeProfile), "profile is not a ResumeProfile instance"
assert isinstance(profile.personal, PersonalInfo), "personal is not PersonalInfo"
assert isinstance(profile.personal.name, str) and profile.personal.name, \
    "name must be a non-empty string"
assert isinstance(profile.personal.email, str) and profile.personal.email, \
    "email must be a non-empty string"
assert isinstance(profile.skills, list), "skills must be a list"
assert isinstance(profile.work, list), "work must be a list"
assert isinstance(profile.education, list), "education must be a list"
assert len(profile.work) > 0, "expected at least one work experience entry"

# Confirm each WorkExperience has typed year fields
for job in profile.work:
    assert isinstance(job.start_year, int), \
        f"start_year must be int, got {type(job.start_year)} for {job.company}"

print("Day 9 gate: PASSED")
print("Deliverable: fully validated ResumeProfile printed as indented JSON above.")
print("All five Day 9 concepts demonstrated:")
print("  [1] Schema-guided extraction (Define-Prompt-Validate)")
print("  [2] Optional fields, nested models, Field(description=...)")
print("  [3] Few-shot grounding (two example pairs per section)")
print("  [4] Extract-validate-retry loop")
print("  [5] Section-aware extraction (split, extract, assemble)")